# Prosperity Round 3 — ML-Driven Alpha Mining

**目标**：用 ML 从大量候选因子里筛出 OOS 稳定的 alpha，输出可直接复制到 `trader_round3.py` 的常量+权重。

**改进 vs 旧版**：
1. 数据清洗：跨日 per-day demean、outlier mask、crossed book 剔除
2. Smile 拟合：剔除 K=4500（边界点污染），用 Huber 鲁棒回归，输出 per-day diagnostics
3. 因子库扩展到 ~40 个：微结构 / 动量 / 波动 / MR / OFI / 成交流 / 跨品种 / IV 衍生
4. ML 筛选：LightGBM gain importance + permutation importance + per-fold stability
5. 线性输出：Ridge 给出 OOS R²>0 的 top-K 因子权重
6. Voucher 残差 alpha：smile residual 是否 mean-revert + voucher 流是否预测 VEV

**用法**：上传 6 个 CSV → 顺序跑 → 末 cell 输出配置块。

In [ ]:
# === Cell 1: 安装依赖 + 上传 ===
!pip install -q lightgbm scikit-learn pandas numpy scipy matplotlib seaborn
from google.colab import files
print('上传 6 个 CSV: prices_round_3_day_{0,1,2}.csv 和 trades_round_3_day_{0,1,2}.csv')
uploaded = files.upload()
print('已上传:', list(uploaded.keys()))

In [ ]:
# === Cell 2: 加载 + 清洗 ===
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

DAY_LEN = 1_000_000

prices_dfs, trades_dfs = [], []
for d in [0, 1, 2]:
    p = pd.read_csv(f'prices_round_3_day_{d}.csv', sep=';')
    p['day'] = d
    prices_dfs.append(p)
    t = pd.read_csv(f'trades_round_3_day_{d}.csv', sep=';')
    t['day'] = d
    trades_dfs.append(t)
prices_raw = pd.concat(prices_dfs, ignore_index=True)
trades_raw = pd.concat(trades_dfs, ignore_index=True)
prices_raw['t_global'] = prices_raw['day'] * DAY_LEN + prices_raw['timestamp']
trades_raw['t_global'] = trades_raw['day'] * DAY_LEN + trades_raw['timestamp']

# === 清洗规则 ===
# (a) bid_price_1 / ask_price_1 / mid_price 必须存在
# (b) 不允许 crossed book (bid_1 >= ask_1)
# (c) 不允许 spread 超过 该 product 全样本 99% 分位 × 3 (极端 outlier)
# (d) bid_volume_1 + ask_volume_1 必须 > 0
# (e) 同一 (product, t_global) 取最后一条 (去重)

def clean_prices(df):
    df = df.copy()
    df = df.dropna(subset=['bid_price_1', 'ask_price_1', 'mid_price'])
    df = df[df['ask_price_1'] > df['bid_price_1']]
    df['spread'] = df['ask_price_1'] - df['bid_price_1']
    df = df.groupby('product', group_keys=False).apply(
        lambda g: g[g['spread'] <= g['spread'].quantile(0.99) * 3]
    )
    df = df[(df['bid_volume_1'].fillna(0) + df['ask_volume_1'].fillna(0)) > 0]
    df = df.sort_values(['product', 't_global']).drop_duplicates(['product', 't_global'], keep='last')
    return df.reset_index(drop=True)

prices = clean_prices(prices_raw)
trades = trades_raw.sort_values('t_global').reset_index(drop=True)

print(f'Prices raw: {len(prices_raw):,} → cleaned: {len(prices):,} ({100*len(prices)/len(prices_raw):.1f}%)')
print(f'Trades: {len(trades):,}')
print(f'Products: {sorted(prices["product"].unique())}')
print(f'\n每个 product 各天 row 数:')
print(prices.groupby(['product', 'day']).size().unstack(fill_value=0))

In [ ]:
# === Cell 3: per-day FAIR/STD（demean 后）===
# 关键：直接 pool 计算 std 会被跨日漂移污染。
# 正确做法：每日单独算 mean/std，然后取 mean(daily_means) 作为 fair，
# sqrt(mean(daily_var)) 作为 std（去掉 inter-day drift）。

def per_day_fair_std(df, product):
    sub = df[df['product'] == product]
    daily = sub.groupby('day')['mid_price'].agg(['mean', 'std', 'count', 'min', 'max'])
    fair_pool   = sub['mid_price'].mean()
    std_pool    = sub['mid_price'].std()
    fair_recent = daily['mean'].iloc[-1]                      # 最近一天 mean，最贴 Round 3 开盘
    fair_avg    = daily['mean'].mean()                        # 三天均值
    std_intra   = np.sqrt((daily['std'] ** 2 * daily['count']).sum() / daily['count'].sum())
    return daily, fair_pool, std_pool, fair_recent, fair_avg, std_intra

for prod in ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT']:
    daily, fp, sp, fr, fa, si = per_day_fair_std(prices, prod)
    print(f'\n=== {prod} ===')
    print(daily.round(2))
    print(f'pooled mean = {fp:.2f}   pooled std = {sp:.2f}')
    print(f'recent-day mean = {fr:.2f}   3-day-avg mean = {fa:.2f}')
    print(f'intra-day std (pure)  = {si:.2f}   ← 用这个，避免 inter-day drift 污染')
    print(f'ratio std_pool / std_intra = {sp/si:.3f}   (>1.05 说明日间漂移显著)')

# 保存供 cell 10 使用
_, _, _, HYDROGEL_FAIR_REC, _, HYDROGEL_STD_INTRA = per_day_fair_std(prices, 'HYDROGEL_PACK')
_, _, _, VEV_FAIR_REC,      _, VEV_STD_INTRA      = per_day_fair_std(prices, 'VELVETFRUIT_EXTRACT')

In [ ]:
# === Cell 4: Smile 拟合（清洗版）===
# 改进：
# (1) 剔除 K=4500（边界点 m≈-1，污染抛物线）
# (2) 剔除 price <= intrinsic + 0.5（IV 反推不稳）
# (3) Huber 鲁棒回归（抗 outlier）
# (4) 报 per-day 系数，看 smile 是否漂移
from scipy.stats import norm
from sklearn.linear_model import HuberRegressor

TTE_DAYS = {0: 8, 1: 7, 2: 6}
STRIKES_FIT = [5000, 5100, 5200, 5300, 5400, 5500]   # 排除 4500/4000/6000/6500

def implied_vol_vec(prices_arr, S, K, T, n_iter=60):
    prices_arr = np.asarray(prices_arr, dtype=float)
    S = np.asarray(S, dtype=float)
    T = np.asarray(T, dtype=float)
    intrinsic = np.maximum(S - K, 0.0)
    bad = (prices_arr <= intrinsic + 0.5) | (T <= 0)
    def bs(sigma_arr):
        sqrtT = np.sqrt(T)
        with np.errstate(divide='ignore', invalid='ignore'):
            d1 = (np.log(S/K) + 0.5*sigma_arr*sigma_arr*T) / (sigma_arr*sqrtT)
            d2 = d1 - sigma_arr*sqrtT
            val = S*norm.cdf(d1) - K*norm.cdf(d2)
        return np.where((T>0) & (sigma_arr>0), val, intrinsic)
    lo = np.full_like(prices_arr, 1e-4); hi = np.full_like(prices_arr, 5.0)
    for _ in range(n_iter):
        midv = 0.5*(lo+hi); fmid = bs(midv) - prices_arr
        up = fmid < 0; lo = np.where(up, midv, lo); hi = np.where(up, hi, midv)
    iv = 0.5*(lo+hi); iv[bad] = np.nan
    return iv

mid_pivot = prices.pivot_table(index=['day','timestamp'], columns='product', values='mid_price').reset_index()
S_arr = mid_pivot['VELVETFRUIT_EXTRACT'].values
day_arr = mid_pivot['day'].astype(int).values
T_arr = np.array([TTE_DAYS[d] / 252.0 for d in day_arr])
ts_arr = mid_pivot['timestamp'].values

rows = []
for K in STRIKES_FIT:
    col = f'VEV_{K}'
    if col not in mid_pivot.columns: continue
    P = mid_pivot[col].values
    valid = ~(np.isnan(S_arr) | np.isnan(P))
    if valid.sum() == 0: continue
    iv = implied_vol_vec(P[valid], S_arr[valid], K, T_arr[valid])
    m  = np.log(K / S_arr[valid]) / np.sqrt(T_arr[valid])
    rows.append(pd.DataFrame({
        'day': day_arr[valid], 'timestamp': ts_arr[valid],
        'K': K, 'S': S_arr[valid], 'T': T_arr[valid],
        'price': P[valid], 'iv': iv, 'm': m,
    }))
iv_df = pd.concat(rows, ignore_index=True).dropna(subset=['iv'])
print(f'IV obs: {len(iv_df):,}   m range: [{iv_df["m"].min():.2f}, {iv_df["m"].max():.2f}]')

# Huber 鲁棒拟合 (主输出)
X_full = np.column_stack([iv_df['m']**2, iv_df['m']])
huber = HuberRegressor().fit(X_full, iv_df['iv'])
a, b = huber.coef_; c = huber.intercept_
iv_df['fit_iv'] = a*iv_df['m']**2 + b*iv_df['m'] + c
iv_df['resid']  = iv_df['iv'] - iv_df['fit_iv']
print(f'\nHuber smile: IV(m) = {a:.5f}*m^2 + {b:.5f}*m + {c:.5f}')
print(f'Residual std (cleaned): {iv_df["resid"].std():.6f}')

# Per-day 检查 smile 漂移
print('\nPer-day smile (a, b, c):')
for d in [0,1,2]:
    sub = iv_df[iv_df['day']==d]
    if len(sub) < 20: continue
    Xs = np.column_stack([sub['m']**2, sub['m']])
    h_d = HuberRegressor().fit(Xs, sub['iv'])
    print(f'  day {d}: a={h_d.coef_[0]:.5f}  b={h_d.coef_[1]:.5f}  c={h_d.intercept_:.5f}  (n={len(sub)})')
print('\nPer-strike residual:')
print(iv_df.groupby('K')['resid'].agg(['mean','std','count']).round(5))

# 可视化
fig, ax = plt.subplots(figsize=(11,5))
for d in [0,1,2]:
    sub = iv_df[iv_df['day']==d]
    ax.scatter(sub['m'], sub['iv'], s=2, alpha=0.25, label=f'day {d}')
mm = np.linspace(iv_df['m'].min(), iv_df['m'].max(), 100)
ax.plot(mm, a*mm**2 + b*mm + c, 'r-', lw=2, label='Huber global')
ax.set_xlabel('m = ln(K/S)/√T'); ax.set_ylabel('IV'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Cleaned IV Smile (K=4500/4000/6000/6500 excluded)'); plt.show()

SMILE_A, SMILE_B, SMILE_C = a, b, c
SMILE_RESID_STD = iv_df['resid'].std()

In [ ]:
# === Cell 5: 因子库 (factor zoo) ===
# 8 类共 ~40 个因子。每个因子都通过 pandas vectorized 实现，避免循环。
# 类别：
#   A) 微结构    : spread, micro_price, depth, imbalance (multi-level)
#   B) 动量      : multi-horizon mid diff
#   C) 波动      : rolling std, vol-of-vol
#   D) 均值回归  : z-score from rolling mean, expanding mean
#   E) OFI       : Order Flow Imbalance (Cont 2014)
#   F) 成交流    : 用 trade price vs mid 推测 buy/sell-initiated 净流量
#   G) 跨品种    : HYDROGEL/VEV ratio z-score、滞后传染
#   H) IV 衍生  : 仅适用于现货预测——voucher 总成交流 × delta 推测对 underlying 的压力

def compute_ofi(p):
    """Order Flow Imbalance per Cont (2014) using L1 book changes."""
    db = p['bid_price_1'].diff()
    da = p['ask_price_1'].diff()
    bv = p['bid_volume_1'].fillna(0); bv_lag = bv.shift(1).fillna(0)
    av = p['ask_volume_1'].fillna(0); av_lag = av.shift(1).fillna(0)
    e_b = np.where(db > 0, bv,
          np.where(db < 0, -bv_lag, bv - bv_lag))
    e_a = np.where(da > 0, -av_lag,
          np.where(da < 0, av, av - av_lag))
    return pd.Series(e_b - e_a, index=p.index)

def signed_trade_flow(trades_df, prices_df, product):
    """Lee-Ready：trade price > 当前 mid → buy-initiated (+qty)，否则 sell-initiated (-qty)。
    返回按 t_global 聚合的净 signed quantity 序列。"""
    t = trades_df[trades_df['symbol'] == product][['t_global','price','quantity']].copy()
    if len(t) == 0:
        return pd.Series(dtype=float)
    p = prices_df[prices_df['product'] == product][['t_global','mid_price']].sort_values('t_global')
    t = t.sort_values('t_global')
    merged = pd.merge_asof(t, p, on='t_global', direction='backward')
    merged['sign'] = np.sign(merged['price'] - merged['mid_price']).fillna(0)
    merged['signed_qty'] = merged['sign'] * merged['quantity']
    return merged.groupby('t_global')['signed_qty'].sum()

def voucher_delta_pressure(trades_df, prices_df, S_series, T):
    """对每张 voucher 算 signed_qty × delta，加总作为 underlying 的 implied 流量。"""
    pressure = pd.Series(0.0, index=S_series.index)
    for K in STRIKES_FIT:
        prod = f'VEV_{K}'
        sf = signed_trade_flow(trades_df, prices_df, prod)
        if len(sf) == 0: continue
        sf = sf.reindex(S_series.index).fillna(0)
        # delta 用 smile IV 算
        with np.errstate(divide='ignore', invalid='ignore'):
            m = np.log(K / S_series.values) / np.sqrt(T)
            sigma = SMILE_A*m**2 + SMILE_B*m + SMILE_C
            d1 = (np.log(S_series.values/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
            delta = norm.cdf(d1)
        pressure = pressure + sf.values * delta
    return pressure

def build_factors(prices_df, trades_df, product, cross_mid=None):
    p = prices_df[prices_df['product']==product].sort_values('t_global').copy()
    p = p.drop_duplicates('t_global', keep='last').set_index('t_global')
    p['mid'] = p['mid_price']

    # A) 微结构
    p['spread'] = p['ask_price_1'] - p['bid_price_1']
    bv1 = p['bid_volume_1'].fillna(0); av1 = p['ask_volume_1'].fillna(0)
    bv2 = p['bid_volume_2'].fillna(0); av2 = p['ask_volume_2'].fillna(0)
    bv3 = p['bid_volume_3'].fillna(0); av3 = p['ask_volume_3'].fillna(0)
    denom1 = (bv1 + av1).replace(0, np.nan)
    p['micro_price'] = (p['bid_price_1']*av1 + p['ask_price_1']*bv1) / denom1
    p['micro_dev']   = p['micro_price'] - p['mid']
    p['imb_l1']  = (bv1 - av1) / (bv1 + av1 + 1e-6)
    p['imb_l2']  = ((bv1+bv2) - (av1+av2)) / ((bv1+bv2)+(av1+av2) + 1e-6)
    p['imb_all'] = ((bv1+bv2+bv3) - (av1+av2+av3)) / ((bv1+bv2+bv3)+(av1+av2+av3) + 1e-6)
    p['depth_l1'] = bv1 + av1
    p['depth_total'] = (bv1+bv2+bv3) + (av1+av2+av3)
    p['bid2_dist'] = p['bid_price_1'] - p['bid_price_2'].fillna(p['bid_price_1'])
    p['ask2_dist'] = p['ask_price_2'].fillna(p['ask_price_1']) - p['ask_price_1']

    # B) 动量
    for w in [3, 5, 10, 20, 50, 100, 200]:
        p[f'mom_{w}'] = p['mid'].diff(w)

    # C) 波动
    ret = p['mid'].diff()
    for w in [10, 20, 50, 100]:
        p[f'vol_{w}'] = ret.rolling(w).std()
    p['vov_50'] = p['vol_20'].rolling(50).std()

    # D) Mean reversion
    for w in [50, 100, 200, 500]:
        rm = p['mid'].rolling(w).mean()
        rs = p['mid'].rolling(w).std().replace(0, np.nan)
        p[f'zsc_{w}'] = (p['mid'] - rm) / rs
    long_mean = p['mid'].expanding(min_periods=200).mean()
    p['dev_long'] = p['mid'] - long_mean

    # E) OFI
    ofi = compute_ofi(p)
    for w in [10, 50, 200]:
        p[f'ofi_{w}'] = ofi.rolling(w).sum()

    # F) 成交流（own product）
    sf = signed_trade_flow(trades_df, prices_df, product)
    sf = sf.reindex(p.index).fillna(0)
    for w in [10, 50, 200]:
        p[f'sflow_{w}'] = sf.rolling(w).sum()
    p['sflow_cum'] = sf.cumsum()

    # G) 跨品种（cross_mid 是另一个产品 mid 序列，已对齐到 p.index）
    if cross_mid is not None:
        cm = cross_mid.reindex(p.index).ffill()
        ratio = p['mid'] / cm
        for w in [50, 200]:
            rm = ratio.rolling(w).mean()
            rs = ratio.rolling(w).std().replace(0, np.nan)
            p[f'cross_z_{w}'] = (ratio - rm) / rs
        for w in [5, 20, 100]:
            p[f'cross_mom_{w}'] = cm.diff(w)

    return p

# 构造 HYDROGEL / VEV 因子（VEV 还要加 voucher delta pressure）
vev_mid_series = prices[prices['product']=='VELVETFRUIT_EXTRACT'].set_index('t_global')['mid_price']
hyd_mid_series = prices[prices['product']=='HYDROGEL_PACK'].set_index('t_global')['mid_price']

feat_h   = build_factors(prices, trades, 'HYDROGEL_PACK', cross_mid=vev_mid_series)
feat_vev = build_factors(prices, trades, 'VELVETFRUIT_EXTRACT', cross_mid=hyd_mid_series)

# H) Voucher delta pressure 仅作用于 VEV
T_for_vev = pd.Series([TTE_DAYS[(idx // DAY_LEN)] / 252.0 for idx in feat_vev.index], index=feat_vev.index).values
vev_S_series = feat_vev['mid']
press = voucher_delta_pressure(trades, prices, vev_S_series, T_for_vev)
for w in [10, 50, 200]:
    feat_vev[f'voucher_pressure_{w}'] = press.rolling(w).sum()
feat_vev['voucher_pressure_cum'] = press.cumsum()

FACTOR_PREFIXES = ('mom_','vol_','imb_','micro','dev_','spread','depth','bid2','ask2',
                   'zsc_','ofi_','sflow_','cross_','vov_','voucher_pressure')
f_cols_h   = [c for c in feat_h.columns   if c.startswith(FACTOR_PREFIXES)]
f_cols_vev = [c for c in feat_vev.columns if c.startswith(FACTOR_PREFIXES)]
print(f'HYDROGEL feats: {feat_h.shape[1]} cols, factor cols = {len(f_cols_h)}')
print(f'VEV feats     : {feat_vev.shape[1]} cols, factor cols = {len(f_cols_vev)}')

In [ ]:
# === Cell 6: 标签 + 多 horizon LightGBM 重要性 ===
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

HORIZONS = [5, 20, 100]

def add_labels(feat, horizons=HORIZONS):
    feat = feat.copy()
    for h in horizons:
        feat[f'y_{h}'] = feat['mid'].shift(-h) - feat['mid']
    return feat

feat_h   = add_labels(feat_h)
feat_vev = add_labels(feat_vev)

def lgbm_importance_cv(feat, factor_cols, label, n_splits=5):
    df = feat[factor_cols + [label]].replace([np.inf, -np.inf], np.nan).dropna()
    X, y = df[factor_cols].values, df[label].values
    tscv = TimeSeriesSplit(n_splits=n_splits)
    imp = np.zeros(len(factor_cols))
    fold_imp = []
    r2s = []
    for tr, te in tscv.split(X):
        m = lgb.LGBMRegressor(
            n_estimators=400, learning_rate=0.04, num_leaves=31,
            min_child_samples=80, reg_lambda=1.5, subsample=0.8,
            colsample_bytree=0.8, verbose=-1)
        m.fit(X[tr], y[tr])
        fi = m.feature_importances_.astype(float)
        fi = fi / (fi.sum() + 1e-9)
        imp += fi
        fold_imp.append(fi)
        r2s.append(r2_score(y[te], m.predict(X[te])))
    imp /= n_splits
    fold_imp = np.array(fold_imp)
    # 稳定性：所有 fold 都进入 top-15 的因子
    in_top = (np.argsort(-fold_imp, axis=1)[:, :15])
    stable = []
    for j in range(len(factor_cols)):
        ranks_in = sum(j in in_top[k] for k in range(n_splits))
        if ranks_in >= n_splits - 1:    # 至少 4/5 fold 在 top-15
            stable.append(factor_cols[j])
    return pd.Series(imp, index=factor_cols).sort_values(ascending=False), r2s, stable

results = {}
for name, feat, fcols in [('HYDROGEL', feat_h, f_cols_h), ('VEV', feat_vev, f_cols_vev)]:
    print(f'\n========== {name} ==========')
    for h in HORIZONS:
        imp_s, r2s, stable = lgbm_importance_cv(feat, fcols, f'y_{h}', n_splits=5)
        results[(name, h)] = {'imp': imp_s, 'r2s': r2s, 'stable': stable}
        print(f'\n--- horizon={h} | mean OOS R²={np.mean(r2s):.4f}  per fold {[round(r,4) for r in r2s]}')
        print(f'    Top 12 importance:')
        for k, v in imp_s.head(12).items():
            mark = '★' if k in stable else ' '
            print(f'      {mark} {k:30s}  imp={v:.4f}')
        print(f'    Stable across folds (≥4/5 in top-15): {stable}')

In [ ]:
# === Cell 7: 在 stable factors 上跑 Ridge，得到可直接复制的线性权重 ===
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def fit_linear_top(feat, stable_factors, label, alpha=1.0, train_frac=0.7):
    if len(stable_factors) == 0:
        print(f'    no stable factors for {label}, skip')
        return {}
    df = feat[stable_factors + [label]].replace([np.inf,-np.inf], np.nan).dropna()
    if len(df) < 1000:
        print(f'    not enough samples ({len(df)}) for {label}, skip')
        return {}
    X = df[stable_factors].values; y = df[label].values
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    sp = int(len(df) * train_frac)
    m = Ridge(alpha=alpha).fit(Xs[:sp], y[:sp])
    yhat_oos = m.predict(Xs[sp:])
    r2_oos = r2_score(y[sp:], yhat_oos)
    coef_unscaled = m.coef_ / sc.scale_
    weights = dict(zip(stable_factors, coef_unscaled))
    print(f'    OOS R² = {r2_oos:.5f}  intercept = {m.intercept_:.4e}')
    print(f'    weights (unscaled, ready to copy):')
    for k, v in sorted(weights.items(), key=lambda kv: -abs(kv[1])):
        print(f'      {k:30s}  {v:+.4e}')
    return {'weights': weights, 'intercept': m.intercept_, 'r2_oos': r2_oos}

# 选每个 product 在所有 horizon 中并集的稳定因子（更鲁棒）
linear_results = {}
for name, feat in [('HYDROGEL', feat_h), ('VEV', feat_vev)]:
    union_stable = sorted(set().union(*[results[(name, h)]['stable'] for h in HORIZONS]))
    print(f'\n========== {name} ==========')
    print(f'union stable factors ({len(union_stable)}): {union_stable}')
    for h in HORIZONS:
        print(f'\n--- horizon={h} Ridge fit ---')
        linear_results[(name, h)] = fit_linear_top(feat, union_stable, f'y_{h}')

In [ ]:
# === Cell 8: Voucher 残差 alpha — smile residual 是否 mean-revert? ===
# 假设：每张 voucher 的 IV 在 smile 上的残差是平稳的 → 残差大时押反向
# 测试：用残差_t 预测 voucher_price 在 [t, t+h] 的变动

iv_panel = iv_df.copy()
iv_panel['t_global'] = iv_panel['day'] * DAY_LEN + iv_panel['timestamp']
iv_panel = iv_panel.sort_values(['K', 't_global'])

# 把 voucher mid 拼接进来 (per K)
v_mids = {}
for K in STRIKES_FIT:
    sub = prices[prices['product']==f'VEV_{K}'].sort_values('t_global')
    v_mids[K] = sub.set_index('t_global')['mid_price']

rows = []
for K in STRIKES_FIT:
    sub = iv_panel[iv_panel['K']==K].set_index('t_global')
    if K not in v_mids: continue
    sub['mkt_price'] = v_mids[K].reindex(sub.index)
    for h in [20, 100, 500]:
        sub[f'fwd_{h}'] = sub['mkt_price'].shift(-h) - sub['mkt_price']
    sub['K'] = K
    rows.append(sub)
iv_with_fwd = pd.concat(rows).dropna(subset=['resid'])

print('=== Smile residual → 未来 voucher 价格变动相关性 ===')
print('（残差为正 = market IV 高于 smile fit；如果 mean-revert，应负相关）')
for h in [20, 100, 500]:
    sub = iv_with_fwd.dropna(subset=[f'fwd_{h}'])
    overall = sub['resid'].corr(sub[f'fwd_{h}'])
    print(f'\n  horizon={h:4d}  overall corr = {overall:+.4f}')
    by_K = sub.groupby('K').apply(lambda g: g['resid'].corr(g[f'fwd_{h}']))
    for K, c in by_K.items():
        print(f'              K={K} corr = {c:+.4f}  (n={len(sub[sub["K"]==K])})')

# 如果残差能预测，把它作为 voucher 的 alpha 直接用
print('\n判断标准：corr 在多数 K 上 < -0.05 → smile residual 可作为 mean-revert signal')
print('实战用法：trader 中每张 voucher 计算 cur_iv - fit_iv，若 > +k*resid_std → 卖；< -k*resid_std → 买。')

In [ ]:
# === Cell 9: voucher_pressure × VEV move — 是否有 lead-lag? ===
# 测试：voucher 大量 buy → 做市商对冲 → VEV 短期上行?

press_series = pd.Series(press, index=feat_vev.index)
vev_mid = feat_vev['mid']
for w in [10, 50, 200]:
    p_w = press_series.rolling(w).sum()
    for h in [5, 20, 100]:
        fwd = vev_mid.shift(-h) - vev_mid
        c = p_w.corr(fwd)
        print(f'  pressure_{w:3d}  →  VEV_fwd_{h:3d}    corr = {c:+.4f}')
print('\n如果某组合 |corr| > 0.05 → voucher_pressure 是 VEV 的有效 alpha，可写入 trader 做 VEV fair 偏置')

In [ ]:
# === Cell 10: 最终配置块（复制到 trader_round3.py 顶部）===
print('# ============================================================')
print('# 复制以下到 trader_round3.py')
print('# ============================================================')
print(f'SMILE_COEF = ({SMILE_A:.6f}, {SMILE_B:.6f}, {SMILE_C:.6f})')
print(f'SMILE_RESID_STD = {SMILE_RESID_STD:.6f}')
print()
print(f'VEV_FAIR = {VEV_FAIR_REC:.1f}        # recent-day mean (clean)')
print(f'VEV_STD  = {VEV_STD_INTRA:.2f}      # intra-day std (no inter-day drift)')
print(f'HYDROGEL_FAIR = {HYDROGEL_FAIR_REC:.1f}')
print(f'HYDROGEL_STD  = {HYDROGEL_STD_INTRA:.2f}')
print()
print('# === ML-mined alpha weights (top stable factors) ===')
for name in ['HYDROGEL','VEV']:
    for h in HORIZONS:
        r = linear_results.get((name, h), {})
        if not r or 'weights' not in r: continue
        if r['r2_oos'] <= 0:
            print(f'# {name}_h{h}: OOS R²={r["r2_oos"]:.4f} ≤ 0 — skip (no edge)')
            continue
        print(f'# {name}_h{h}: OOS R²={r["r2_oos"]:.4f}')
        print(f'ALPHA_{name}_H{h} = {{')
        for k, v in sorted(r["weights"].items(), key=lambda kv: -abs(kv[1])):
            print(f'    {k!r}: {v:+.6e},')
        print(f'}}  # intercept = {r["intercept"]:+.4e}')
        print()